In [ ]:
!pip install -q sentence-transformers scikit-learn pandas matplotlib

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
chunks = [
    "Employees receive 12 casual leaves annually.",
    "Employees receive 15 sick leaves annually.",
    "Employees may work from home twice per week.",
    "Travel expenses are reimbursed within 30 days.",
    "All employees are covered under company medical insurance."
]

#Task 1: Load Embedding Model

In [ ]:
model_name = "all-MiniLM-L6-v2"

model = SentenceTransformer(model_name)

print("Model Name:", model_name)

sample_embedding = model.encode("Sample sentence")

print("Embedding Dimension:", len(sample_embedding))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model Name: all-MiniLM-L6-v2
Embedding Dimension: 384


#Task 2: Generate Embeddings

In [ ]:
chunk_embeddings = model.encode(chunks)

for chunk, emb in zip(chunks, chunk_embeddings):
    print("\nChunk:")
    print(chunk)
    print("Embedding Shape:", emb.shape)


Chunk:
Employees receive 12 casual leaves annually.
Embedding Shape: (384,)

Chunk:
Employees receive 15 sick leaves annually.
Embedding Shape: (384,)

Chunk:
Employees may work from home twice per week.
Embedding Shape: (384,)

Chunk:
Travel expenses are reimbursed within 30 days.
Embedding Shape: (384,)

Chunk:
All employees are covered under company medical insurance.
Embedding Shape: (384,)


#Task 3: Analyze Embedding Vectors

In [ ]:
print("Chunk:")
print(chunks[0])

print("\nFirst 20 Values:")
print(chunk_embeddings[0][:20])

Chunk:
Employees receive 12 casual leaves annually.

First 20 Values:
[ 0.0618362   0.01376683  0.03366624  0.0186107   0.03135883  0.06788085
 -0.01135737 -0.01733116 -0.07070484  0.01901567  0.1098766   0.05092814
 -0.0489678  -0.04620624 -0.03665631  0.00247606 -0.06287521  0.00541349
  0.03131726 -0.07714854]


#Task 4: Generate Query Embeddings

In [ ]:
queries = [
    "How many casual leaves are allowed?",
    "Can employees work remotely?",
    "What is the travel reimbursement process?",
    "Do employees have medical insurance?"
]

query_embeddings = model.encode(queries)

for query, emb in zip(queries, query_embeddings):
    print("\nQuery:")
    print(query)
    print("Embedding Shape:", emb.shape)


Query:
How many casual leaves are allowed?
Embedding Shape: (384,)

Query:
Can employees work remotely?
Embedding Shape: (384,)

Query:
What is the travel reimbursement process?
Embedding Shape: (384,)

Query:
Do employees have medical insurance?
Embedding Shape: (384,)


#Task 5: Semantic Similarity Analysis

In [ ]:
results = []

for q, q_emb in zip(queries, query_embeddings):

    similarities = cosine_similarity(
        [q_emb],
        chunk_embeddings
    )[0]

    for chunk, score in zip(chunks, similarities):

        results.append([
            q,
            chunk,
            round(float(score),4)
        ])

similarity_df = pd.DataFrame(
    results,
    columns=[
        "Query",
        "Chunk",
        "Similarity Score"
    ]
)

similarity_df

,Query,Chunk,Similarity Score
0,How many casual leaves are allowed?,Employees receive 12 casual leaves annually.,0.6552
1,How many casual leaves are allowed?,Employees receive 15 sick leaves annually.,0.4110
2,How many casual leaves are allowed?,Employees may work from home twice per week.,0.1763
3,How many casual leaves are allowed?,Travel expenses are reimbursed within 30 days.,0.0493
4,How many casual leaves are allowed?,All employees are covered under company medica...,0.0472
5,Can employees work remotely?,Employees receive 12 casual leaves annually.,0.1963
6,Can employees work remotely?,Employees receive 15 sick leaves annually.,0.2300
7,Can employees work remotely?,Employees may work from home twice per week.,0.4834
8,Can employees work remotely?,Travel expenses are reimbursed within 30 days.,-0.0254
9,Can employees work remotely?,All employees are covered under company medica...,0.3006


#Task 6: Identify Most Similar Chunk

In [ ]:
for q, q_emb in zip(queries, query_embeddings):

    similarities = cosine_similarity(
        [q_emb],
        chunk_embeddings
    )[0]

    best_index = similarities.argmax()

    print("\n" + "="*60)
    print("Query:")
    print(q)

    print("\nMost Similar Chunk:")
    print(chunks[best_index])

    print(
        "\nSimilarity Score:",
        round(float(similarities[best_index]),4)
    )


Query:
How many casual leaves are allowed?

Most Similar Chunk:
Employees receive 12 casual leaves annually.

Similarity Score: 0.6552

Query:
Can employees work remotely?

Most Similar Chunk:
Employees may work from home twice per week.

Similarity Score: 0.4834

Query:
What is the travel reimbursement process?

Most Similar Chunk:
Travel expenses are reimbursed within 30 days.

Similarity Score: 0.7255

Query:
Do employees have medical insurance?

Most Similar Chunk:
All employees are covered under company medical insurance.

Similarity Score: 0.803


#Task 7: Similar Meaning vs Different Meaning

In [ ]:
pair1 = [
    "Employees receive 12 casual leaves.",
    "Workers are entitled to 12 annual leaves."
]

pair2 = [
    "Employees receive 12 casual leaves.",
    "Travel expenses are reimbursed within 30 days."
]

pair1_emb = model.encode(pair1)
pair2_emb = model.encode(pair2)

sim1 = cosine_similarity(
    [pair1_emb[0]],
    [pair1_emb[1]]
)[0][0]

sim2 = cosine_similarity(
    [pair2_emb[0]],
    [pair2_emb[1]]
)[0][0]

print("Pair 1 Similarity:", round(float(sim1),4))
print("Pair 2 Similarity:", round(float(sim2),4))

Pair 1 Similarity: 0.771
Pair 2 Similarity: 0.175
